# VoiceStudio: train your personal voice (free GPU)

This notebook turns the dataset you recorded in the VoiceStudio app into a `.voicepack`
you can import on your phone. After that the voice works **fully offline** on the phone;
this notebook is the only step that uses the internet.

**Before you start**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive, create a folder `VoiceStudio` and upload into it:
   - your dataset from the app (`something.voicedataset.zip`)
   - the folder `tools/voice-training` from the VoiceStudio project (as `VoiceStudio/voice-training`)
3. Run the cells from top to bottom.

Only train voices of people who gave their permission. The dataset contains the consent
record confirmed in the app; training stops if it is missing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings
`BASE = "bangla"` gives the best Bangla pronunciation (publish with a short credit line).
`BASE = "ljspeech"` uses only public-domain data plus your recordings (no credit needed), but Bangla
can sound less natural with small datasets.

`EPOCHS`: 1000 is a good start for 30–60 minutes of recordings (about 2–4 hours on a T4).
Training is saved to Google Drive, so if Colab disconnects, run the **Resume** cell.

In [ ]:
DATASET = "/content/drive/MyDrive/VoiceStudio/my-voice.voicedataset.zip"  # <- your file name
KIT     = "/content/drive/MyDrive/VoiceStudio/voice-training"
RUN_DIR = "/content/drive/MyDrive/VoiceStudio/runs/my-voice"
BASE    = "bangla"       # "bangla" or "ljspeech"
EPOCHS  = 1000
INT8    = False          # True = smaller file, slightly lower quality

import os
assert os.path.isfile(DATASET), f"Dataset not found: {DATASET}"
assert os.path.isfile(f"{KIT}/train.py"), f"Training kit not found in {KIT}"

## 1. Install the trainer (about 3 minutes)

In [ ]:
%%bash
set -e
cd /content
if [ ! -d piper1-gpl ]; then
  git clone --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git
fi
cd piper1-gpl
pip install -q -e '.[train]'
./build_monotonic_align.sh
python setup.py build_ext --inplace -q
pip install -q "sherpa-onnx==1.13.8" onnx onnxruntime soundfile
echo "Trainer ready"

## 2. Check and prepare your recordings

In [ ]:
!cd "$KIT" && python prepare_dataset.py --dataset "$DATASET" --out /content/data

## 3. Train
Progress is printed as it goes. Checkpoints are written to `RUN_DIR` on your Drive.
When training finishes, the voice is exported and checked automatically.

In [ ]:
int8 = "--int8" if INT8 else ""
!cd "$KIT" && python train.py --data /content/data --base "$BASE" --run-dir "$RUN_DIR" --epochs $EPOCHS $int8 --out "$RUN_DIR/my-voice.voicepack"

### Resume (only if Colab disconnected)
Run the install and prepare cells again, then this one. Raise `EPOCHS` to train longer.

In [ ]:
int8 = "--int8" if INT8 else ""
!cd "$KIT" && python train.py --data /content/data --run-dir "$RUN_DIR" --epochs $EPOCHS --resume $int8 --out "$RUN_DIR/my-voice.voicepack"

## 4. Listen and download
Below are test sentences in each style. If they sound right, download the `.voicepack`
(it is also in your Drive at `RUN_DIR`), copy it to your phone and import it in
**VoiceStudio → Voices → Import**.

In [ ]:
import glob
from IPython.display import Audio, display
for wav in sorted(glob.glob(f"{RUN_DIR}/export/samples/*.wav")):
    print(os.path.basename(wav))
    display(Audio(wav))

In [ ]:
from google.colab import files
files.download(f"{RUN_DIR}/my-voice.voicepack")